# Hospital Readmissions Reduction Program Analysis

## Analysis Objective

This project examines CMS Hospital Readmission Reduction Program data to identify patterns in excess readmission ratios (ERR) and determine whether above-expected readmission performance is concentrated amaong a subset of hospitals.

An ERR above 1.0 indicates that a hospital's risk-adjusted readmissions for a condition were higher than expected, while an ERR below 1.0 indicates lower-than-expected readmissions.

The analyis asks two primary questions:

1. Do Medical and Surgical HRRP conditions show meaningfully different ERR patterns?
2. Is above-expected readmission performance disproportionately concentrated among a smaller group of facilities?

## 1. Data Loading and Validation

In [1]:
import pandas as pd

df = pd.read_csv(
    "../data/raw/readmission_reduction.csv",
    dtype={"Facility ID": "string"}
)

In [2]:
df.shape

(18510, 12)

### Initial Raw Data Checks

**Why this validation matters:** Facility ID is an identifier rather than a numeric measurement, so it was loaded as a string to preserve formatting such as leading zeros. I also checked ID lengths, facility counts, measure counts, and availability of ERR values before beginning the analysis.

In [3]:
df["Facility ID"].dtype

StringDtype

In [4]:
print("Facility ID dtype:", df["Facility ID"].dtype)
print("\nFirst 5 Facility IDs:")
print(df["Facility ID"].head())

print("\nFacility ID lengths:")
print(df["Facility ID"].str.len().value_counts().sort_index())

Facility ID dtype: string

First 5 Facility IDs:
0    010001
1    010001
2    010001
3    010001
4    010001
Name: Facility ID, dtype: string

Facility ID lengths:
6    18510
Name: Facility ID, dtype: Int64


In [5]:
print("Unique facilities:", df["Facility ID"].nunique())

print("\nMeasure counts:")
print(df["Measure Name"].value_counts())

print("\nRows with usable ERR:",
      df["Excess Readmission Ratio"].notna().sum())

Unique facilities: 3085

Measure counts:
READM-30-HIP-KNEE-HRRP    3085
READM-30-COPD-HRRP        3085
READM-30-PN-HRRP          3085
READM-30-AMI-HRRP         3085
READM-30-HF-HRRP          3085
READM-30-CABG-HRRP        3085
Name: Measure Name, dtype: int64

Rows with usable ERR: 11927


### Condition Mapping and Validation

In [6]:
condition_map = {
    "READM-30-AMI-HRRP": "Medical",
    "READM-30-HF-HRRP": "Medical",
    "READM-30-PN-HRRP": "Medical",
    "READM-30-COPD-HRRP": "Medical",
    "READM-30-CABG-HRRP": "Surgical",
    "READM-30-HIP-KNEE-HRRP": "Surgical"
}

expected_measures = set(condition_map.keys())
actual_measures = set(df["Measure Name"].dropna().unique())

assert actual_measures == expected_measures, (
    f"Measure mismatch. "
    f"Missing from mapping: {actual_measures - expected_measures}. "
    f"Unexpected in mapping: {expected_measures - actual_measures}."
)

print("Validation passed: all CMS measures are included in the condition mapping.")

Validation passed: all CMS measures are included in the condition mapping.


**Data-quality validation**

During an earlier version of the analysis, I identified an inconsistency in the condition-group counts. Investigation showed that the AMI measure code did not match the value used in my mapping logic, causing 1,763 usable AMI observations to be excluded from the Medical group.

I corrected the measure mapping and reran the affected analysis. I then added explicit validation checks to confirm that every measure appearing in the CMS dataset exists in the mapping and that every record receives a condition type.

These checks are retained below so that a future change in the source data or mapping cannot silently exclude a condition from the analysis.

In [7]:
df["Condition Type"] = df["Measure Name"].map(condition_map)

print("Condition Type counts:")
print(df["Condition Type"].value_counts())

print("\nUnmapped records:", df["Condition Type"].isna().sum())

assert df["Condition Type"].notna().all(), (
    "Validation failed: one or more records were not assigned a Condition Type."
)

print("\nValidation passed: every record was assigned a Condition Type.")

Condition Type counts:
Medical     12340
Surgical     6170
Name: Condition Type, dtype: int64

Unmapped records: 0

Validation passed: every record was assigned a Condition Type.


## 2. Analysis Dataset

**Creating the analytical dataset**

ERR is the primary metric used throughout this analysis, so records without a reported ERR cannot contribute to the statistical or facility-level comparisons below. I therefore created a seperate analytical dataset containing only records with usable ERR values while leaving the original dataframe unchanged.

This reduces the datset from 18,510 hospital-condition records to 11,927  usable records across 2,862 facilities.

In [8]:
analysis_df = df.dropna(subset=["Excess Readmission Ratio"]).copy()

print("Usable ERR records:", len(analysis_df))
print("Unique facilities:", analysis_df["Facility ID"].nunique())

print("\nCondition Type counts:")
print(analysis_df["Condition Type"].value_counts())

print("\nUnmapped records:",
      analysis_df["Condition Type"].isna().sum())

Usable ERR records: 11927
Unique facilities: 2862

Condition Type counts:
Medical     9456
Surgical    2471
Name: Condition Type, dtype: int64

Unmapped records: 0


### Descriptive Statistics by Condition Type

In [9]:
summary = (
    analysis_df
    .groupby("Condition Type")["Excess Readmission Ratio"]
    .agg(["mean", "median", "std", "count"])
)

print(summary)

                    mean  median       std  count
Condition Type                                   
Medical         1.001469  0.9986  0.064060   9456
Surgical        1.002678  0.9946  0.124963   2471


**Analytical question:** One possible explanation for differences in readmission performance is the broad type of condition being measured. I therefore compared ERR between the four Medical measures and two Surgical measures.

Because the groups differ substantially in sample size and may not have equal variances, I used Welch's two-sample t-test rather than assuming equal variance.

Null hypothesis: Mean ERR is equal between Medical and Surgical conditions.

Alternative hypothesis: Mean ERR differs between the two groups.

### Welch's Two-Sample T-Test

In [10]:
from scipy.stats import ttest_ind

medical = analysis_df.loc[
    analysis_df["Condition Type"] == "Medical",
    "Excess Readmission Ratio"
]

surgical = analysis_df.loc[
    analysis_df["Condition Type"] == "Surgical",
    "Excess Readmission Ratio"
]

t_stat, p_value = ttest_ind(
    medical,
    surgical,
    equal_var=False
)

print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")

T-statistic: -0.4650
P-value: 0.6419


Medical and Surgical records had very similar mean ERR values (1.0015 vs. 1.0027). Welch's t-test produced a p-value of 0.6419, so this analysis does not provide evidence of a statistically significant difference in mean ERR between the two broad condition groups.

Because the Medical/Surgical classification did not explain much of the variation in ERR, I shifted the analysis toward facility-level patterns to determine whether above-expected readmission performance was concentrated among particular hospitals.

### Records Above Expected Readmission Levels

In [11]:
analysis_df["Above Expected"] = (
    analysis_df["Excess Readmission Ratio"] > 1
)

print(analysis_df["Above Expected"].value_counts())

print("\nPercent above expected:")
print(
    analysis_df["Above Expected"].value_counts(normalize=True) * 100
)

False    6113
True     5814
Name: Above Expected, dtype: int64

Percent above expected:
False    51.253459
True     48.746541
Name: Above Expected, dtype: float64


In [12]:
facility_summary = (
    analysis_df
    .groupby(["Facility ID", "Facility Name"])
    .agg(
        usable_conditions=("Excess Readmission Ratio", "count"),
        above_expected_conditions=("Above Expected", "sum"),
        mean_err=("Excess Readmission Ratio", "mean")
    )
    .reset_index()
)

facility_summary.head()

,Facility ID,Facility Name,usable_conditions,above_expected_conditions,mean_err
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,6,1,0.971467
1,010005,MARSHALL MEDICAL CENTERS,4,0,0.890575
2,010006,NORTH ALABAMA MEDICAL CENTER,6,2,0.992550
3,010007,MIZELL MEMORIAL HOSPITAL,4,3,1.028350
4,010011,ST. VINCENT'S EAST,6,4,1.025167


In [13]:
facility_distribution = (
    facility_summary["above_expected_conditions"]
    .value_counts()
    .sort_index()
)

print(facility_distribution)

0    487
1    681
2    639
3    561
4    334
5    124
6     36
Name: above_expected_conditions, dtype: int64


In [14]:
facilities_3plus = (
    facility_summary["above_expected_conditions"] >= 3
).sum()

total_facilities = len(facility_summary)

percent_3plus = (
    facilities_3plus / total_facilities * 100
)

print("Facilities with 3+ conditions above expected:", facilities_3plus)
print("Total facilities:", total_facilities)
print(f"Percent with 3+ conditions above expected: {percent_3plus:.2f}%")

Facilities with 3+ conditions above expected: 1055
Total facilities: 2862
Percent with 3+ conditions above expected: 36.86%


### Concentration of Above-Expected ERR Records Across Facilities

In [15]:
facility_concentration = (
    facility_summary
    .sort_values("above_expected_conditions", ascending=False)
    .copy()
)

total_above_expected = facility_concentration["above_expected_conditions"].sum()

facility_concentration["cumulative_above_expected"] = (
    facility_concentration["above_expected_conditions"].cumsum()
)

facility_concentration["cumulative_share"] = (
    facility_concentration["cumulative_above_expected"]
    / total_above_expected
    * 100
)

facilities_for_80 = (
    facility_concentration["cumulative_share"] < 80
).sum() + 1

percent_facilities_for_80 = (
    facilities_for_80 / len(facility_concentration) * 100
)

print("Total above-expected condition records:", total_above_expected)
print("Facilities needed to account for 80%:", facilities_for_80)
print(
    f"Share of facilities needed to account for 80%: "
    f"{percent_facilities_for_80:.2f}%"
)

Total above-expected condition records: 5814
Facilities needed to account for 80%: 1454
Share of facilities needed to account for 80%: 50.80%


**Moving from frequency to magnitude**

Counting conditions above ERR 1.0 identifies how frequently a facility performs above expected readmission levels, but it treats an ERR of 1.01 the same as an ERR of 1.20.

To incorporate the magnitude of above-expected performance, I created an analytical measure called ERR Excess, defined as:

max(ERR - 1.0, 0)

This assigns zero to records at or below expected levels and measures only the amount by which an above-expected record exceeds 1.0.

This is a project-specific prioritization metric, not an official CMS penalty or financial measure. Its purpose is to determine whether the magnitude of above-expected ERR is concentrated among a subset of hospitals.

### Concentration of ERR Excess Above 1.0

In [16]:
analysis_df["ERR Excess"] = (
    analysis_df["Excess Readmission Ratio"] - 1
).clip(lower=0)

facility_excess = (
    analysis_df
    .groupby(["Facility ID", "Facility Name"])
    .agg(total_err_excess=("ERR Excess", "sum"))
    .reset_index()
    .sort_values("total_err_excess", ascending=False)
)

total_err_excess = facility_excess["total_err_excess"].sum()

facility_excess["cumulative_excess"] = (
    facility_excess["total_err_excess"].cumsum()
)

facility_excess["cumulative_share"] = (
    facility_excess["cumulative_excess"]
    / total_err_excess
    * 100
)

facilities_for_80_excess = (
    facility_excess["cumulative_share"] < 80
).sum() + 1

percent_facilities_for_80_excess = (
    facilities_for_80_excess / len(facility_excess) * 100
)

print("Total ERR excess above 1.0:", round(total_err_excess, 4))
print("Facilities needed to account for 80% of ERR excess:",
      facilities_for_80_excess)
print(
    f"Share of facilities needed to account for 80% of ERR excess: "
    f"{percent_facilities_for_80_excess:.2f}%"
)

Total ERR excess above 1.0: 357.8068
Facilities needed to account for 80% of ERR excess: 1057
Share of facilities needed to account for 80% of ERR excess: 36.93%


**Result:** 1,057 of the 2,862 facilities in the analytical dataset (36.93%) account for 80% of the aggregate positive ERR excess.

This suggests that the magnitude of above-expected readmission performance is not evenly distributed across facilities. For a quality-improvement team, this type of prioritization could help narrow an initial review toward facilities contributing disproportionately to the observed above-expected ERR before investigating the underlying causes.

The analysis does not establish why those hospitals have higher ERR values, so facility-level investigation would be required before recommending specific interventions.

## Key Findings and Limitations

### Key findings

Medical and Surgical condition groups had very similar average ERR values, and Welch's t-test did not identify a statistically significant difference between them (p = 0.6419).
5,814 of 11,927 usable hospital-condition records (48.7%) had ERR values above 1.0.
1,055 of 2,862 facilities (36.9%) had at least three conditions with above-expected ERR.
When both frequency and magnitude were considered, 1,057 facilities (36.93%) accounted for 80% of aggregate positive ERR excess.

### Business interpretation

The results suggest that broad Medical versus Surgical classification provides limited differentiation, while facility-level analysis reveals stronger concentration. A quality-improvement team could use this type of analysis as an initial screening tool to prioritize hospitals for deeper investigation rather than reviewing all facilities equally.

### Limitations

ERR excess is an analytical prioritization measure created for this project and is not an official CMS financial penalty measure.
The concentration calculation is not weighted by patient volume or number of eligible discharges.
Facility-level ERR data can identify patterns but cannot establish the causes of readmissions.
This analysis uses aggregated hospital-condition data rather than patient-level clinical data.
Additional operational, demographic, financial, and patient-level information would be required before recommending specific interventions.